In [ ]:
pip install yfinance

In [1]:
import yfinance as yf
import pandas as pd

stocks = ["TCS.NS", "INFY.NS", "RELIANCE.NS"]

all_data = []

for stock in stocks:

    df = yf.download(
        stock,
        start="2015-01-01",
        end="2025-01-01"
    )
    
    # Flatten multi-level columns returned by newer yfinance versions
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    # reset date from index into column
    df.reset_index(inplace=True)

    # add company column
    df["Company"] = stock

    # keep only needed columns
    df = df[[
        "Date",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume",
        "Company"
    ]]

    all_data.append(df)

# vertical concatenation
fdf = pd.concat(all_data, ignore_index=True)

print(fdf.head())

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Price       Date         Open         High         Low        Close   Volume  \
0     2015-01-01   997.212507   997.212507  987.112186   988.879761   366830   
1     2015-01-02   990.996947  1006.516461  990.841596  1002.049011   925740   
2     2015-01-05  1002.651236  1009.993355  980.760691   986.820923  1754242   
3     2015-01-06   982.489405   982.489405  947.876342   950.440308  2423784   
4     2015-01-07   959.530459   963.084957  935.231400   939.213257  2636332   

Price Company  
0      TCS.NS  
1      TCS.NS  
2      TCS.NS  
3      TCS.NS  
4      TCS.NS  


In [2]:
fdf.head()


Price,Date,Open,High,Low,Close,Volume,Company
0,2015-01-01,997.212507,997.212507,987.112186,988.879761,366830,TCS.NS
1,2015-01-02,990.996947,1006.516461,990.841596,1002.049011,925740,TCS.NS
2,2015-01-05,1002.651236,1009.993355,980.760691,986.820923,1754242,TCS.NS
3,2015-01-06,982.489405,982.489405,947.876342,950.440308,2423784,TCS.NS
4,2015-01-07,959.530459,963.084957,935.231400,939.213257,2636332,TCS.NS


In [3]:
fdf.describe()

Price,Date,Open,High,Low,Close,Volume
count,7401,7401.000000,7401.000000,7401.000000,7401.000000,7.401000e+03
mean,2020-01-01 11:22:56.084312832,1245.142100,1257.315225,1232.407698,1244.679800,9.704301e+06
min,2015-01-01 00:00:00,174.742513,174.977893,170.430652,173.490677,8.682200e+04
25%,2017-07-04 00:00:00,521.302830,529.351649,510.893989,519.613098,2.963643e+06
50%,2020-01-07 00:00:00,1025.109827,1036.283602,1014.884475,1023.988708,6.621839e+06
75%,2022-06-30 00:00:00,1541.168021,1560.630532,1523.994406,1542.458252,1.241994e+07
max,2024-12-31 00:00:00,4332.652012,4348.038052,4272.055480,4311.585449,1.644050e+08
std,NaN,930.352829,938.584286,922.016410,930.206001,1.099946e+07


In [4]:
fdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7401 entries, 0 to 7400
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   Date     7401 non-null   datetime64[ns]
 1   Open     7401 non-null   float64       
 2   High     7401 non-null   float64       
 3   Low      7401 non-null   float64       
 4   Close    7401 non-null   float64       
 5   Volume   7401 non-null   int64         
 6   Company  7401 non-null   object        
dtypes: datetime64[ns](1), float64(4), int64(1), object(1)
memory usage: 404.9+ KB


In [5]:
fdf.isnull().sum()

Price
Date       0
Open       0
High       0
Low        0
Close      0
Volume     0
Company    0
dtype: int64

## Feature Engg

Adding column RETURN

In [6]:
fdf["Return"] = fdf.groupby("Company")[
    "Close"
].pct_change()

Adding column MOVING AVGS

In [7]:
fdf["MA10"] = fdf.groupby("Company")[
    "Close"
].transform(lambda x: x.rolling(10).mean())

fdf["MA50"] = fdf.groupby("Company")[
    "Close"
].transform(lambda x: x.rolling(50).mean())

In [8]:
fdf.head()

Price,Date,Open,High,Low,Close,Volume,Company,Return,MA10,MA50
0,2015-01-01,997.212507,997.212507,987.112186,988.879761,366830,TCS.NS,NaN,NaN,NaN
1,2015-01-02,990.996947,1006.516461,990.841596,1002.049011,925740,TCS.NS,0.013317,NaN,NaN
2,2015-01-05,1002.651236,1009.993355,980.760691,986.820923,1754242,TCS.NS,-0.015197,NaN,NaN
3,2015-01-06,982.489405,982.489405,947.876342,950.440308,2423784,TCS.NS,-0.036866,NaN,NaN
4,2015-01-07,959.530459,963.084957,935.231400,939.213257,2636332,TCS.NS,-0.011812,NaN,NaN


adding column VOLATILITY

In [9]:
fdf["Volatility"] = fdf.groupby(
    "Company"
)["Return"].transform(
    lambda x: x.rolling(10).std()
)

adding column DAILY MOMENTUM

In [10]:
fdf["Momentum"] = fdf.groupby(
    "Company"
)["Close"].diff(5)

adding Column RSI(>70 overbought & <30 oversold)

In [11]:
delta = fdf.groupby("Company")[
    "Close"
].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()

rs = avg_gain / avg_loss

fdf["RSI"] = 100 - (100 / (1 + rs))

fdf["Range"] = (
    fdf["High"] - fdf["Low"]
)

fdf["MA_Ratio"] = (
    fdf["MA10"] /
    fdf["MA50"]
)




In [12]:
fdf.describe()

Price,Date,Open,High,Low,Close,Volume,Return,MA10,MA50,Volatility,Momentum,RSI,Range,MA_Ratio
count,7401,7401.000000,7401.000000,7401.000000,7401.000000,7.401000e+03,7398.000000,7374.000000,7254.000000,7371.000000,7386.000000,7359.000000,7401.000000,7254.000000
mean,2020-01-01 11:22:56.084312832,1245.142100,1257.315225,1232.407698,1244.679800,9.704301e+06,0.000786,1244.020617,1240.455522,0.014775,3.706765,53.245907,24.907527,1.013677
min,2015-01-01 00:00:00,174.742513,174.977893,170.430652,173.490677,8.682200e+04,-0.161881,177.740459,185.221896,0.003087,-362.447144,7.280559,1.435262,0.742670
25%,2017-07-04 00:00:00,521.302830,529.351649,510.893989,519.613098,2.963643e+06,-0.007898,524.140487,536.407031,0.010518,-13.995678,40.898160,10.033030,0.983141
50%,2020-01-07 00:00:00,1025.109827,1036.283602,1014.884475,1023.988708,6.621839e+06,0.000576,1023.240204,1015.517293,0.013338,2.671234,53.565828,18.217283,1.014496
75%,2022-06-30 00:00:00,1541.168021,1560.630532,1523.994406,1542.458252,1.241994e+07,0.009396,1544.566971,1538.500444,0.017102,22.193352,65.545327,32.465668,1.043169
max,2024-12-31 00:00:00,4332.652012,4348.038052,4272.055480,4311.585449,1.644050e+08,0.147180,4273.281787,4149.701792,0.090672,383.896240,97.544032,270.897176,1.215321
std,NaN,930.352829,938.584286,922.016410,930.206001,1.099946e+07,0.016380,928.300471,919.151169,0.007206,49.779193,16.533030,22.037990,0.049695


In [13]:
fdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7401 entries, 0 to 7400
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Date        7401 non-null   datetime64[ns]
 1   Open        7401 non-null   float64       
 2   High        7401 non-null   float64       
 3   Low         7401 non-null   float64       
 4   Close       7401 non-null   float64       
 5   Volume      7401 non-null   int64         
 6   Company     7401 non-null   object        
 7   Return      7398 non-null   float64       
 8   MA10        7374 non-null   float64       
 9   MA50        7254 non-null   float64       
 10  Volatility  7371 non-null   float64       
 11  Momentum    7386 non-null   float64       
 12  RSI         7359 non-null   float64       
 13  Range       7401 non-null   float64       
 14  MA_Ratio    7254 non-null   float64       
dtypes: datetime64[ns](1), float64(12), int64(1), object(1)
memory usage: 867

In [14]:
fdf=fdf.dropna()
fdf.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7254 entries, 49 to 7400
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Date        7254 non-null   datetime64[ns]
 1   Open        7254 non-null   float64       
 2   High        7254 non-null   float64       
 3   Low         7254 non-null   float64       
 4   Close       7254 non-null   float64       
 5   Volume      7254 non-null   int64         
 6   Company     7254 non-null   object        
 7   Return      7254 non-null   float64       
 8   MA10        7254 non-null   float64       
 9   MA50        7254 non-null   float64       
 10  Volatility  7254 non-null   float64       
 11  Momentum    7254 non-null   float64       
 12  RSI         7254 non-null   float64       
 13  Range       7254 non-null   float64       
 14  MA_Ratio    7254 non-null   float64       
dtypes: datetime64[ns](1), float64(12), int64(1), object(1)
memory usage: 906.8+ 

##### extracting date and month (donno why but still doing it)

In [15]:
fdf["Month"] = (
    fdf["Date"].dt.month
)

fdf["DayOfWeek"] = (
    fdf["Date"].dt.dayofweek
)

### creating target variable 
if next day close is > today's close its 1 else 0.

In [16]:
fdf["Target"] = (
    fdf.groupby("Company")[
        "Close"
    ].shift(-1) >
    fdf["Close"]
).astype(int)

#### encoding the companies

In [17]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

fdf["Company_Code"] = (
    encoder.fit_transform(
        fdf["Company"]
    )
)

In [18]:
fdf.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7254 entries, 49 to 7400
Data columns (total 19 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Date          7254 non-null   datetime64[ns]
 1   Open          7254 non-null   float64       
 2   High          7254 non-null   float64       
 3   Low           7254 non-null   float64       
 4   Close         7254 non-null   float64       
 5   Volume        7254 non-null   int64         
 6   Company       7254 non-null   object        
 7   Return        7254 non-null   float64       
 8   MA10          7254 non-null   float64       
 9   MA50          7254 non-null   float64       
 10  Volatility    7254 non-null   float64       
 11  Momentum      7254 non-null   float64       
 12  RSI           7254 non-null   float64       
 13  Range         7254 non-null   float64       
 14  MA_Ratio      7254 non-null   float64       
 15  Month         7254 non-null   int32       

### adding nlp sentiment part 
using marketaux api for news as marketaux provides comprehensive market data, including historical company news ,Also it can handle requests in large volume. and finbert for sentiment analysis as its specifically trained on finanial words so it can more accurately sort bullish and bearish.


this is gonna be a full nlp pipeline . from cleaning to tokenization to final sentiment that will be used as a feature in the above dataset.

In [19]:
ndf=pd.read_csv(r"C:\Users\arije\Downloads\archive (5)\stock_news_2016 to 2026.csv")
ndf.head()

,date,title,description,url,source_file,categories,matched_keywords,relevance_score,has_negation,impact_tier
0,2016-01-01,Lending to foreign step-down arms of Indian fi...,RBI slaps 2% additional provision for such loans,URL_NOT_AVAILABLE,IndianFinancialNews.csv,sector_banking_finance,rbi,2,False,LOW
1,2016-01-02,IDBI Bank to raise Rs 900 cr through Basel-III...,Many banks have been raising money through tie...,URL_NOT_AVAILABLE,IndianFinancialNews.csv,sector_banking_finance,basel,2,False,LOW
2,2016-01-02,Forex reserves up $943 mn,India's foreign exchange reserves rose $943 mi...,URL_NOT_AVAILABLE,IndianFinancialNews.csv,macro_government,forex reserve,3,False,MEDIUM
3,2016-01-02,SBI: Lending rate cut unlikely till end-Mar,Country's largest lender SBI today ruled out f...,URL_NOT_AVAILABLE,IndianFinancialNews.csv,stock_specific,sbi,2,True,LOW
4,2016-01-03,"ASK Group to invest Rs 1,500 cr in real estate...",ASK Group plans to step up its equity investme...,URL_NOT_AVAILABLE,IndianFinancialNews.csv,sector_cement_infra,real estate,1,False,LOW


In [20]:
ndf.drop(columns=["url","source_file","categories"], inplace=True)

In [21]:
df.describe()   

Price,Date,Open,High,Low,Close,Volume
count,2467,2467.000000,2467.000000,2467.000000,2467.000000,2.467000e+03
mean,2020-01-01 11:22:56.084313088,731.223435,738.686447,723.345324,730.697571,1.842886e+07
min,2015-01-01 00:00:00,174.742513,174.977893,170.430652,173.490677,1.705656e+06
25%,2017-07-04 12:00:00,315.861159,317.608704,313.160518,315.653030,1.069616e+07
50%,2020-01-07 00:00:00,645.396234,652.978941,636.583452,642.041199,1.464381e+07
75%,2022-06-29 12:00:00,1122.798986,1133.629472,1112.048646,1121.365723,2.109197e+07
max,2024-12-31 00:00:00,1592.662006,1596.980044,1573.851280,1589.138062,1.426834e+08
std,NaN,420.153243,423.754484,416.275051,419.850320,1.355224e+07


In [22]:
keywords = [
    "TCS",
    "Tata Consultancy",
    "Infosys",
    "Reliance"
]

pattern = "|".join(keywords)

In [23]:
filtered_news = ndf[
    ndf["title"]
    .str.contains(
        pattern,
        case=False,
        na=False
    )
]

In [24]:
print(filtered_news.shape)

print(filtered_news.head())

(6397, 7)
            date                                              title  \
146   2016-02-11  CCI approves Nippon Life stake hike in Relianc...   
978   2016-09-07  Reliance Capital raises Rs 2,000 cr though deb...   
979   2016-09-07  Reliance Capital raises ~2,000 cr though deben...   
1000  2016-09-13  Reliance Capital to list home finance biz; eye...   
1032  2016-09-21          Reliance Nippon Life bullish on UP market   

                                            description matched_keywords  \
146   Nippon Life had announced its plan to hike the...         reliance   
978   Funds will be utilised for refinancing existin...         reliance   
979   The finance company will use the proceeds to r...         reliance   
1000  All shareholders of Reliance Capital will rece...         reliance   
1032  Insurer to ramp up advisors' network to 25,000...         reliance   

      relevance_score  has_negation impact_tier  
146                 2         False         LOW  
978   

In [25]:
def assign_company(headline):

    headline = headline.lower()

    if "tcs" in headline or \
       "tata consultancy" in headline:

        return "TCS.NS"

    elif "infosys" in headline:

        return "INFY.NS"

    elif "reliance" in headline:

        return "RELIANCE.NS"

    else:
        return None

In [26]:
filtered_news["Company"] = (
    filtered_news["title"]
    .apply(assign_company)
)

C:\Users\arije\AppData\Local\Temp\ipykernel_4860\2828545426.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_news["Company"] = (


In [27]:
filtered_news.dropna(
    subset=["Company"],
    inplace=True
)

C:\Users\arije\AppData\Local\Temp\ipykernel_4860\2600320720.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_news.dropna(


In [28]:
print(
    filtered_news["Company"]
    .value_counts()
)

Company
RELIANCE.NS    2968
TCS.NS         1847
INFY.NS        1582
Name: count, dtype: int64


In [29]:
filtered_news.head()    

,date,title,description,matched_keywords,relevance_score,has_negation,impact_tier,Company
146,2016-02-11,CCI approves Nippon Life stake hike in Relianc...,Nippon Life had announced its plan to hike the...,reliance,2,False,LOW,RELIANCE.NS
978,2016-09-07,"Reliance Capital raises Rs 2,000 cr though deb...",Funds will be utilised for refinancing existin...,reliance,2,False,LOW,RELIANCE.NS
979,2016-09-07,"Reliance Capital raises ~2,000 cr though deben...",The finance company will use the proceeds to r...,reliance,2,False,LOW,RELIANCE.NS
1000,2016-09-13,Reliance Capital to list home finance biz; eye...,All shareholders of Reliance Capital will rece...,reliance,2,False,LOW,RELIANCE.NS
1032,2016-09-21,Reliance Nippon Life bullish on UP market,"Insurer to ramp up advisors' network to 25,000...",reliance,2,False,LOW,RELIANCE.NS


since the dataset is ready now we use FinBERT.


In [30]:
from transformers import pipeline
classifier = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [31]:
headline = filtered_news.iloc[0]["title"]

print(headline)

CCI approves Nippon Life stake hike in Reliance Life to 49%


In [32]:
result = classifier(headline)

print(result)

[{'label': 'positive', 'score': 0.8691171407699585}]


In [33]:
def get_sentiment_score(text):

    result = classifier(text)[0]

    label = result["label"]
    score = result["score"]

    if label == "positive":
        return score

    elif label == "negative":
        return -score

    else:
        return 0

In [34]:
sample_news = filtered_news.head(20).copy()
sample_news["Sentiment"] = (
    sample_news["title"]
    .apply(get_sentiment_score)

)
print(
    sample_news[
        ["title", "Sentiment"]
    ]
)

                                                  title  Sentiment
146   CCI approves Nippon Life stake hike in Relianc...   0.869117
978   Reliance Capital raises Rs 2,000 cr though deb...   0.000000
979   Reliance Capital raises ~2,000 cr though deben...   0.000000
1000  Reliance Capital to list home finance biz; eye...   0.000000
1032          Reliance Nippon Life bullish on UP market   0.427561
1036  Andhra Bank ties up with Cigna TTK, Reliance G...   0.000000
1199  Reliance Home Finance plans to garner Rs 3,500...   0.576012
1309  Opening bell : Asia opens mixed ; SBI , Infosy...   0.686420
1340  Sensex , Nifty likely to be flat ; Infosys , a...   0.561786
1413  Sensex , Nifty open flat ; Tata Motors , Infos...   0.894817
1424  Lack of investments in new tech may crimp TCS ...  -0.943057
1467  Sensex rises over 200 pts , Nifty strong ; TCS...   0.931673
1542  Sensex , Nifty fall for 2nd day ; TCS rebounds...  -0.685387
1584  Nifty above 8250 , Sensex climbs over 100 pts ...   0.90

In [38]:

filtered_news["Sentiment"] = (
    filtered_news["title"]
    .apply(get_sentiment_score)
)

C:\Users\arije\AppData\Local\Temp\ipykernel_4860\2793774136.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_news["Sentiment"] = (


In [39]:
filtered_news.to_csv(
    "news_with_sentiment.csv",
    index=False
)

In [41]:
sentiment_df = pd.read_csv(
    "news_with_sentiment.csv"
)
sentiment_df = sentiment_df[
    ["date", "Company", "Sentiment"]
]

In [42]:
sentiment_df.head() 

,date,Company,Sentiment
0,2016-02-11,RELIANCE.NS,0.869117
1,2016-09-07,RELIANCE.NS,0.000000
2,2016-09-07,RELIANCE.NS,0.000000
3,2016-09-13,RELIANCE.NS,0.000000
4,2016-09-21,RELIANCE.NS,0.427561


In [43]:
fdf.rename(columns={"Date": "date"}, inplace=True)

In [44]:
fdf.head()

Price,date,Open,High,Low,Close,Volume,Company,Return,MA10,MA50,Volatility,Momentum,RSI,Range,MA_Ratio,Month,DayOfWeek,Target,Company_Code
49,2015-03-16,1006.292233,1015.945682,993.369111,997.378418,1556296,TCS.NS,-0.007764,1033.368542,998.498488,0.017373,-32.891479,33.799510,22.576571,1.034922,3,0,1,2
50,2015-03-17,1005.241963,1014.661830,995.607988,1005.709045,1267318,TCS.NS,0.008353,1030.032770,998.835073,0.017695,-22.965271,35.857399,19.053841,1.031234,3,1,0,2
51,2015-03-18,1008.530384,1011.839019,993.252274,996.230103,1255170,TCS.NS,-0.009425,1021.599652,998.718695,0.009192,-18.372803,36.557606,18.586745,1.022910,3,2,1,2
52,2015-03-19,1002.711229,1014.330417,999.636182,1010.593567,2448964,TCS.NS,0.014418,1015.770630,999.194148,0.011519,-8.875122,43.173232,14.694235,1.016590,3,3,1,2
53,2015-03-20,1010.496143,1019.059670,1003.820523,1016.315430,1963908,TCS.NS,0.005662,1012.442529,1000.511650,0.011089,11.132507,44.285029,15.239147,1.011925,3,4,1,2


### now merging the two datasets

problem occured with name.(so renamed Date as date). then date column was of different datatypes in the two datasets. one of object and other of date-time

In [ ]:
# fdf = pd.merge(
#     fdf,
#     sentiment_df 
# ,
#     on=["date", "Company"],
#     how="left"
# )

In [ ]:
fdf.info()

made a mistake of not creating the aggregate datset so the overlapping vlaues in the join were too less.

In [46]:
daily_sentiment = (
    sentiment_df.groupby(
        ["date", "Company"]
    )["Sentiment"]
    .mean()
    .reset_index()
)

In [47]:
daily_sentiment.head()

,date,Company,Sentiment
0,2016-02-11,RELIANCE.NS,0.869117
1,2016-09-07,RELIANCE.NS,0.000000
2,2016-09-13,RELIANCE.NS,0.000000
3,2016-09-21,RELIANCE.NS,0.427561
4,2016-09-23,RELIANCE.NS,0.000000


In [52]:
daily_sentiment["date"] = pd.to_datetime(
    daily_sentiment["date"])

In [53]:
print(fdf["date"].dtype)

print(daily_sentiment["date"].dtype)

datetime64[ns]
datetime64[ns]


In [54]:
fdf = pd.merge(
    fdf,
    daily_sentiment 
,
    on=["date", "Company"],
    how="left"
)

In [56]:
fdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7254 entries, 0 to 7253
Data columns (total 20 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          7254 non-null   datetime64[ns]
 1   Open          7254 non-null   float64       
 2   High          7254 non-null   float64       
 3   Low           7254 non-null   float64       
 4   Close         7254 non-null   float64       
 5   Volume        7254 non-null   int64         
 6   Company       7254 non-null   object        
 7   Return        7254 non-null   float64       
 8   MA10          7254 non-null   float64       
 9   MA50          7254 non-null   float64       
 10  Volatility    7254 non-null   float64       
 11  Momentum      7254 non-null   float64       
 12  RSI           7254 non-null   float64       
 13  Range         7254 non-null   float64       
 14  MA_Ratio      7254 non-null   float64       
 15  Month         7254 non-null   int32   

In [58]:
fdf["Sentiment"] = (fdf["Sentiment"].fillna(0))

Dont know much about finance but this advice from gpt seemed fair " to shift teh sentiment scores by one day to avoid temporal leakage"


In [59]:
fdf["Sentiment"] = (fdf.groupby("Company")["Sentiment"].shift(1))
fdf.dropna(inplace=True)

In [60]:
features = [
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "Return",
    "MA10",
    "MA50",
    "Volatility",
    "Momentum",
    "RSI",
    "Range",
    "MA_Ratio",
    "Month",
    "DayOfWeek",
    "Company_Code",
    "Sentiment"
]

In [61]:
X=fdf[features]
y=fdf["Target"]

1    0
2    1
3    1
4    1
5    0
Name: Target, dtype: int64

#### we cant just split randomly so we do a careful chronological split so we dont have a temporal leak.

In [65]:
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]

X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]

y_test = y.iloc[split_index:]

svm cares about scaling , rf and xgboost not too much

In [66]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)

### training Random forest

In [91]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

# Define parameter distribution for Random Forest
rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False],
    'random_state': [42]
}

# Create RF classifier with RandomizedSearchCV
rf_search = RandomizedSearchCV(
    RandomForestClassifier(),
    rf_params,
    n_iter=20,
    cv=5,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

rf = rf_search.fit(X_train, y_train)
print("Best RF Parameters:", rf.best_params_)
print("Best RF CV Score:", rf.best_score_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best RF Parameters: {'random_state': 42, 'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': None, 'max_depth': 30, 'bootstrap': True}
Best RF CV Score: 0.5012940949688364


In [92]:
rf_pred = rf.predict(X_test)

In [93]:
from sklearn.metrics import accuracy_score

print(
    "RF Accuracy:",
    accuracy_score(y_test, rf_pred)
)

RF Accuracy: 0.5231148162182645


### XG Boost

In [71]:
pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.5/101.7 MB 883.8 kB/s eta 0:01:55
   ---------------------------------------- 0.5/101.7 MB 883.8 kB/s eta 0:01:55
   ---------------------------------------- 0.8/101.7 MB 929.5 kB/s eta 0:01:49
   ---------------------------------------- 1.0/101.7 MB 902.4 kB/s eta 0:01:52
    --------------------------------------- 1.3/101.7 MB 897.1 kB/s eta 0:01:52
    --------------------------------------- 1.6/101.7 MB 957.6 kB/s eta 0:01:45
    --------------------------------------- 2.1/101.7 MB 1.2 MB/s eta 0:01:25
   - -------------------------------------- 2.6/101.7 MB 1.4 MB/s eta 0:01:13
   - -------------------------------------- 3.4/101.7 MB 1.6 MB/s eta 0:01:02
   - -------------------------------------- 3.9/101.7 MB 1.7 MB/s eta 0:0

In [94]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

# Define parameter distribution for XGBoost
xgb_params = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [3, 4, 5, 6, 7, 8],
    'learning_rate': [0.01, 0.02, 0.05, 0.1],
    'subsample': [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9],
    'colsample_bylevel': [0.6, 0.7, 0.8, 0.9],
    'min_child_weight': [1, 2, 3, 4, 5],
    'gamma': [0, 0.1, 0.2, 0.3, 0.4],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [0.8, 1.0, 1.2],
    'random_state': [42]
}

# Create XGB classifier with RandomizedSearchCV
xgb_search = RandomizedSearchCV(
    XGBClassifier(),
    xgb_params,
    n_iter=30,
    cv=5,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

xgb = xgb_search.fit(X_train, y_train)
print("Best XGB Parameters:", xgb.best_params_)
print("Best XGB CV Score:", xgb.best_score_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best XGB Parameters: {'subsample': 0.7, 'reg_lambda': 1.2, 'reg_alpha': 0, 'random_state': 42, 'n_estimators': 200, 'min_child_weight': 2, 'max_depth': 7, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 0.9, 'colsample_bylevel': 0.6}
Best XGB CV Score: 0.5175056223093233


In [95]:
xgb_pred = xgb.predict(X_test)

In [96]:
print(
    "XGB Accuracy:",
    accuracy_score(y_test, xgb_pred)
)

XGB Accuracy: 0.5128836680560819


### SVM

In [97]:
from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV

# Define parameter distribution for SVM
svm_params = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf', 'poly'],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1],
    'degree': [2, 3, 4],
    'coef0': [0.0, 0.1, 1.0],
    'probability': [True],
    'random_state': [42]
}

# Create SVM classifier with RandomizedSearchCV
svm_search = RandomizedSearchCV(
    SVC(),
    svm_params,
    n_iter=20,
    cv=5,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

svm = svm_search.fit(
    X_train_scaled,
    y_train
)
print("Best SVM Parameters:", svm.best_params_)
print("Best SVM CV Score:", svm.best_score_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best SVM Parameters: {'random_state': 42, 'probability': True, 'kernel': 'poly', 'gamma': 'scale', 'degree': 4, 'coef0': 0.1, 'C': 0.1}
Best SVM CV Score: 0.5190053331619868


In [98]:
svm_pred = svm.predict(
    X_test_scaled
)

In [99]:
print(
    "SVM Accuracy:",
    accuracy_score(y_test, svm_pred)
)

SVM Accuracy: 0.5075786282682835


In [79]:
print(fdf["Target"].value_counts(normalize=True))

Target
1    0.518135
0    0.481865
Name: proportion, dtype: float64


### Stacking ensemble training

In [83]:
estimators = [
    ("rf", rf),
    ("xgb", xgb),
    ("svm", svm)
]

In [108]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV

# Define parameter distribution for Logistic Regression (Meta Model)
meta_params = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l2'],
    'solver': ['lbfgs', 'liblinear'],
    'max_iter': [100, 200, 500, 1000],
    'class_weight': ['balanced', None],
    'random_state': [42]
}

# Create meta model with RandomizedSearchCV
meta_search = RandomizedSearchCV(
    LogisticRegression(),
    meta_params,
    n_iter=15,
    cv=5,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print("Tuning meta model...")
meta_search.fit(
    rf_search.predict_proba(X_train),
    y_train
)
meta_model = meta_search.best_estimator_
print("Best Meta Model Parameters:", meta_search.best_params_)
print("Best Meta Model CV Score:", meta_search.best_score_)

Tuning meta model...
Fitting 5 folds for each of 15 candidates, totalling 75 fits
Best Meta Model Parameters: {'solver': 'liblinear', 'random_state': 42, 'penalty': 'l2', 'max_iter': 500, 'class_weight': None, 'C': 10}
Best Meta Model CV Score: 0.999493670886076


In [109]:
from sklearn.ensemble import StackingClassifier

# Create stacking classifier with tuned base models and meta model
stack_model = StackingClassifier(
    estimators=[
        ("rf", rf.best_estimator_),
        ("xgb", xgb.best_estimator_),
        ("svm", svm.best_estimator_)
    ],
    final_estimator=meta_model,
    cv=5
)

In [110]:
print("Training stacking model with tuned parameters...")
stack_model.fit(
    X_train_scaled,
    y_train
)
print("Stacking model training complete!")

Training stacking model with tuned parameters...
Stacking model training complete!


In [111]:
stack_pred = stack_model.predict(
    X_test_scaled
)

In [112]:
print(
    "Stacking Accuracy:",
    accuracy_score(y_test, stack_pred)
)

Stacking Accuracy: 0.5104206138688897


In [113]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        stack_pred
    )
)

              precision    recall  f1-score   support

           0       0.46      0.09      0.15      2544
           1       0.52      0.90      0.66      2734

    accuracy                           0.51      5278
   macro avg       0.49      0.50      0.40      5278
weighted avg       0.49      0.51      0.41      5278



In [114]:
from sklearn.metrics import confusion_matrix

print(
    confusion_matrix(
        y_test,
        stack_pred
    )
)

[[ 230 2314]
 [ 270 2464]]


In [107]:

# Summary of all model results with hyperparameter tuning
print("="*60)
print("HYPERPARAMETER TUNING RESULTS SUMMARY")
print("="*60)

print("\n1. RANDOM FOREST - Best Parameters:")
for param, value in rf.best_params_.items():
    print(f"   {param}: {value}")
print(f"   Best CV Score: {rf.best_score_:.4f}")
print(f"   Test Accuracy: {accuracy_score(y_test, rf_pred):.4f}")

print("\n2. XGBOOST - Best Parameters:")
for param, value in xgb.best_params_.items():
    print(f"   {param}: {value}")
print(f"   Best CV Score: {xgb.best_score_:.4f}")
print(f"   Test Accuracy: {accuracy_score(y_test, xgb_pred):.4f}")

print("\n3. SVM - Best Parameters:")
for param, value in svm.best_params_.items():
    print(f"   {param}: {value}")
print(f"   Best CV Score: {svm.best_score_:.4f}")
print(f"   Test Accuracy: {accuracy_score(y_test, svm_pred):.4f}")

print("\n4. META MODEL (Logistic Regression) - Best Parameters:")
for param, value in meta_search.best_params_.items():
    print(f"   {param}: {value}")
print(f"   Best CV Score: {meta_search.best_score_:.4f}")

print("\n5. STACKING ENSEMBLE - Test Accuracy:")
print(f"   Accuracy: {accuracy_score(y_test, stack_pred):.4f}")

print("\n" + "="*60)

HYPERPARAMETER TUNING RESULTS SUMMARY

1. RANDOM FOREST - Best Parameters:
   random_state: 42
   n_estimators: 100
   min_samples_split: 10
   min_samples_leaf: 2
   max_features: None
   max_depth: 30
   bootstrap: True
   Best CV Score: 0.5013
   Test Accuracy: 0.5231

2. XGBOOST - Best Parameters:
   subsample: 0.7
   reg_lambda: 1.2
   reg_alpha: 0
   random_state: 42
   n_estimators: 200
   min_child_weight: 2
   max_depth: 7
   learning_rate: 0.1
   gamma: 0.3
   colsample_bytree: 0.9
   colsample_bylevel: 0.6
   Best CV Score: 0.5175
   Test Accuracy: 0.5129

3. SVM - Best Parameters:
   random_state: 42
   probability: True
   kernel: poly
   gamma: scale
   degree: 4
   coef0: 0.1
   C: 0.1
   Best CV Score: 0.5190
   Test Accuracy: 0.5076

4. META MODEL (Logistic Regression) - Best Parameters:
   solver: liblinear
   random_state: 42
   penalty: l2
   max_iter: 500
   class_weight: None
   C: 10
   Best CV Score: 0.9995

5. STACKING ENSEMBLE - Test Accuracy:
   Accuracy: 0.5